In [ ]:
!pip install -q timm datasets huggingface_hub scikit-learn matplotlib seaborn

import os
os.makedirs("/kaggle/working", exist_ok=True)

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/content/working//) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
!pip install -q datasets timm scikit-learn matplotlib seaborn

In [ ]:
import timm
import datasets
import sklearn

print("timm:", timm.__version__)
print("datasets:", datasets.__version__)
print("scikit-learn:", sklearn.__version__)

In [ ]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names("mohanty/PlantVillage")

print("Available configurations:")
print(configs)

In [ ]:
import kagglehub
plantdoc_path = kagglehub.dataset_download("nirmalsankalana/plantdoc-dataset")
print(plantdoc_path)

In [ ]:
import os
for root, dirs, files in os.walk(plantdoc_path):
    print(root, dirs[:5])

In [ ]:
from huggingface_hub import hf_hub_download

data_zip = hf_hub_download(
    repo_id="mohanty/PlantVillage",
    filename="data.zip",
    repo_type="dataset"
)

print("Downloaded to:")
print(data_zip)

In [ ]:
import os

size_gb = os.path.getsize(data_zip) / (1024**3)

print(f"data.zip size: {size_gb:.2f} GB")

In [ ]:
import zipfile

with zipfile.ZipFile(data_zip, "r") as z:
    files = z.namelist()

print("Number of files:", len(files))

for f in files[:30]:
    print(f)

In [ ]:
import zipfile

with zipfile.ZipFile(data_zip, "r") as z:
    color_files = [
        f for f in z.namelist()
        if f.startswith("raw/color/") and
        f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

print("Color images:", len(color_files))

In [ ]:
classes = sorted(set(
    f.split("/")[2]
    for f in color_files
    if len(f.split("/")) >= 3
))

print("Number of classes:", len(classes))

for i, cls in enumerate(classes):
    print(i, cls)

In [ ]:
from huggingface_hub import hf_hub_download

train_split = hf_hub_download(
    repo_id="mohanty/PlantVillage",
    filename="splits/color_train.txt",
    repo_type="dataset"
)

test_split = hf_hub_download(
    repo_id="mohanty/PlantVillage",
    filename="splits/color_test.txt",
    repo_type="dataset"
)

print("Train split:", train_split)
print("Test split:", test_split)

In [ ]:
with open(train_split, "r") as f:
    train_lines = f.readlines()

with open(test_split, "r") as f:
    test_lines = f.readlines()

print("Training entries:", len(train_lines))
print("Test entries:", len(test_lines))

print("\nFirst 5 training entries:")
for x in train_lines[:5]:
    print(x.strip())

print("\nFirst 5 test entries:")
for x in test_lines[:5]:
    print(x.strip())

In [ ]:
total = len(train_lines) + len(test_lines)

print("Total:", total)
print("Train:", len(train_lines))
print("Test:", len(test_lines))

print("Train %:", round(len(train_lines) / total * 100, 2))
print("Test %:", round(len(test_lines) / total * 100, 2))

In [ ]:
from collections import Counter

def get_classes(lines):
    result = []

    for line in lines:
        path = line.strip()

        if not path:
            continue

        parts = path.split("/")

        # raw/color/CLASS/image.jpg
        if len(parts) >= 4:
            result.append(parts[2])

    return Counter(result)

train_classes = get_classes(train_lines)
test_classes = get_classes(test_lines)

print("Training classes:", len(train_classes))
print("Test classes:", len(test_classes))

In [ ]:
import zipfile
import os

extract_path = "/kaggle/working/plantvillage"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(data_zip, "r") as z:
    color_files = [
        f for f in z.namelist()
        if f.startswith("raw/color/")
    ]

    print("Files to extract:", len(color_files))

    for f in color_files:
        z.extract(f, extract_path)

print("Extraction complete!")

In [ ]:
color_path = "/kaggle/working/plantvillage/raw/color"

print("Classes:", len(os.listdir(color_path)))

total_images = 0

for cls in os.listdir(color_path):
    class_path = os.path.join(color_path, cls)

    if os.path.isdir(class_path):
        count = len([
            f for f in os.listdir(class_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        total_images += count
        print(f"{cls}: {count}")

print("\nTotal images:", total_images)

In [ ]:
import os

COLOR_DIR = "/kaggle/working/plantvillage/raw/color"

print("Color directory exists:", os.path.exists(COLOR_DIR))

In [ ]:


plantdoc_to_pv = {
    "Corn_Gray_leaf_spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_leaf_blight": "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_rust_leaf": "Corn_(maize)___Common_rust_",
    "Tomato_Early_blight_leaf": "Tomato___Early_blight",
    "Tomato_leaf_late_blight": "Tomato___Late_blight",
    "Tomato_Septoria_leaf_spot": "Tomato___Septoria_leaf_spot",
    "Tomato_mold_leaf": "Tomato___Leaf_Mold",
    "Tomato_leaf_bacterial_spot": "Tomato___Bacterial_spot",
    "Tomato_leaf_mosaic_virus": "Tomato___Tomato_mosaic_virus",
    "Tomato_leaf_yellow_virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato_two_spotted_spider_mites_leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato_leaf": "Tomato___healthy",
    "Potato_leaf_early_blight": "Potato___Early_blight",
    "Potato_leaf_late_blight": "Potato___Late_blight",
    "Apple_Scab_Leaf": "Apple___Apple_scab",
    "Apple_rust_leaf": "Apple___Cedar_apple_rust",
    "Apple_leaf": "Apple___healthy",
    "grape_leaf_black_rot": "Grape___Black_rot",
    "grape_leaf": "Grape___healthy",
    "Strawberry_leaf": "Strawberry___healthy",
    "Squash_Powdery_mildew_leaf": "Squash___Powdery_mildew",
    "Bell_pepper_leaf": "Pepper,_bell___healthy",
    "Bell_pepper_leaf_spot": "Pepper,_bell___Bacterial_spot",
    "Blueberry_leaf": "Blueberry___healthy",
    "Raspberry_leaf": "Raspberry___healthy",
    "Soyabean_leaf": "Soybean___healthy",
    "Peach_leaf": "Peach___healthy",
    "Cherry_leaf": "Cherry_(including_sour)___healthy",
}

import shutil
copied = 0
for split in ["train", "test"]:
    split_dir = os.path.join(plantdoc_path, split)
    for pd_class, pv_class in plantdoc_to_pv.items():
        src_dir = os.path.join(split_dir, pd_class)
        dst_dir = os.path.join(COLOR_DIR, pv_class)
        os.makedirs(dst_dir, exist_ok=True)
        if os.path.isdir(src_dir):
            for f in os.listdir(src_dir):
                src_f = os.path.join(src_dir, f)
                if os.path.isfile(src_f):
                    shutil.copy(src_f, os.path.join(dst_dir, f"pd_{split}_{f}"))
                    copied += 1
print("Copied files:", copied)

In [ ]:
import random
random.seed(42)

all_files = []
for cls in os.listdir(COLOR_DIR):
    cls_dir = os.path.join(COLOR_DIR, cls)
    if os.path.isdir(cls_dir):
        for f in os.listdir(cls_dir):
            all_files.append(f"raw/color/{cls}/{f}")

random.shuffle(all_files)
split_idx = int(len(all_files) * 0.8)

with open("/kaggle/working/new_train.txt", "w") as f:
    f.write("\n".join(all_files[:split_idx]))
with open("/kaggle/working/new_test.txt", "w") as f:
    f.write("\n".join(all_files[split_idx:]))

train_split = "/kaggle/working/new_train.txt"
test_split = "/kaggle/working/new_test.txt"
print("Train:", split_idx, "Test:", len(all_files) - split_idx)

In [ ]:
from PIL import Image
from torch.utils.data import Dataset

class PlantVillageDataset(Dataset):

    def __init__(self, split_file, color_dir, transform=None):
        self.color_dir = color_dir
        self.transform = transform

        with open(split_file, "r") as f:
            self.samples = [
                line.strip()
                for line in f
                if line.strip()
            ]

        # Create class names from folder names
        self.classes = sorted([
            d for d in os.listdir(color_dir)
            if os.path.isdir(os.path.join(color_dir, d))
        ])

        self.class_to_idx = {
            cls: i for i, cls in enumerate(self.classes)
        }

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        relative_path = self.samples[idx]

        # Split file contains:
        # raw/color/ClassName/image.jpg
        parts = relative_path.split("/")

        class_name = parts[2]

        image_path = os.path.join(
            self.color_dir,
            class_name,
            parts[3]
        )

        image = Image.open(image_path).convert("RGB")

        label = self.class_to_idx[class_name]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
train_dataset = PlantVillageDataset(
    split_file=train_split,
    color_dir=COLOR_DIR
)

test_dataset = PlantVillageDataset(
    split_file=test_split,
    color_dir=COLOR_DIR
)

print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Classes:", len(train_dataset.classes))

In [ ]:
image, label = train_dataset[0]

print("Image shape:", image.size)
print("Label:", label)
print("Class:", train_dataset.classes[label])

In [ ]:
import matplotlib.pyplot as plt

image, label = train_dataset[0]

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.title(train_dataset.classes[label])
plt.axis("off")
plt.show()

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.35, hue=0.05),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.85,1.15)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [ ]:
train_dataset = PlantVillageDataset(
    split_file=train_split,
    color_dir=COLOR_DIR,
    transform=train_transform
)

test_dataset = PlantVillageDataset(
    split_file=test_split,
    color_dir=COLOR_DIR,
    transform=test_transform
)

print("Training:", len(train_dataset))
print("Test:", len(test_dataset))
print("Classes:", len(train_dataset.classes))

In [ ]:
image, label = train_dataset[0]

print("Tensor shape:", image.shape)
print("Label:", label)
print("Class:", train_dataset.classes[label])

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

In [ ]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

In [ ]:
from collections import Counter
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

class_counts = torch.zeros(len(train_dataset.classes))
for path in train_dataset.samples:
    parts = path.strip().split("/")
    cls = parts[2]
    class_counts[train_dataset.class_to_idx[cls]] += 1

class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights = class_weights.to(device)

print("Min weight:", class_weights.min().item(), "Max weight:", class_weights.max().item())

In [ ]:
import timm
import torch
import torch.nn as nn

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = len(train_dataset.classes)

model = timm.create_model(
    "convnext_tiny",
    pretrained=True,
    num_classes=NUM_CLASSES
)

model = model.to(device)
print("Device:", device)
print("Number of classes:", NUM_CLASSES)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,        # lower than EfficientNet's 1e-4
    weight_decay=1e-4
)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [ ]:
scaler = torch.amp.GradScaler("cuda")

In [ ]:
from sklearn.metrics import f1_score

def evaluate(model, loader, device):
    model.eval()

    all_labels = []
    all_predictions = []

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)

            loss = criterion(outputs, labels)

            predictions = torch.argmax(outputs, dim=1)

            total_loss += loss.item() * images.size(0)
            total_samples += images.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    avg_loss = total_loss / total_samples

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )

    return avg_loss, macro_f1, all_labels, all_predictions

In [ ]:
import time
import copy

EPOCHS = 25
PATIENCE = 5

best_f1 = 0.0
best_model_state = None
epochs_without_improvement = 0

history = {
    "train_loss": [],
    "train_acc": [],
    "test_loss": [],
    "test_f1": []
}

for epoch in range(EPOCHS):

    start_time = time.time()

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # Mixed precision
        with torch.autocast(device_type="cuda", dtype=torch.float16):

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation
    test_loss, test_f1, _, _ = evaluate(
        model,
        test_loader,
        device
    )

    # Scheduler uses macro-F1
    scheduler.step(test_f1)

    # Save history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_f1"].append(test_f1)

    elapsed = time.time() - start_time

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Train Acc: {train_acc:.4f} "
        f"| Val Loss: {test_loss:.4f} "
        f"| Val Macro-F1: {test_f1:.4f} "
        f"| Time: {elapsed/60:.1f} min"
    )

    # Best model
    if test_f1 > best_f1:
      best_f1 = test_f1
      best_model_state = copy.deepcopy(model.state_dict())
      torch.save(best_model_state, "/kaggle/working/checkpoint_latest.pth")  # add this line
      epochs_without_improvement = 0
    else:

        epochs_without_improvement += 1

        print(
            f"  No improvement "
            f"({epochs_without_improvement}/{PATIENCE})"
        )

    # Early stopping
    if epochs_without_improvement >= PATIENCE:

        print("Early stopping triggered.")

        break

In [ ]:

import os
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
model.load_state_dict(best_model_state)
model.eval()

print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print(f"Best Validation Macro-F1: {best_f1:.4f}")
print(f"Best Validation Macro-F1: {best_f1 * 100:.2f}%")
print()

In [ ]:
checkpoint = {
    "model_name": "convnext_tiny",
    "model_state_dict": model.state_dict(),

    "class_names": train_dataset.classes,
    "class_to_idx": train_dataset.class_to_idx,

    "num_classes": NUM_CLASSES,
    "image_size": 224,

    "best_macro_f1": best_f1,

    "history": history
}

save_path = "/kaggle/working/convnext_tiny_best.pth"

torch.save(checkpoint, save_path)

print("=" * 60)
print("MODEL SAVED")
print("=" * 60)
print("Path:", save_path)
print("Size:",
      round(os.path.getsize(save_path) / (1024 * 1024), 2),
      "MB")
print()

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    range(1, len(history["train_loss"]) + 1),
    history["train_loss"],
    marker="o",
    label="Training Loss"
)

plt.plot(
    range(1, len(history["test_loss"]) + 1),
    history["test_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("convnext_tiny — Training vs Validation Loss")

plt.legend()
plt.grid(True)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    range(1, len(history["test_f1"]) + 1),
    history["test_f1"],
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("convnext_tiny — Validation Macro-F1")

plt.grid(True)
plt.tight_layout()

plt.show()

In [ ]:
model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        y_true.extend(
            labels.cpu().numpy()
        )

        y_pred.extend(
            predictions.cpu().numpy()
        )

In [ ]:
accuracy = accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted"
)

print("=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

print(f"Accuracy:       {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Macro-F1:       {macro_f1:.4f} ({macro_f1 * 100:.2f}%)")
print(f"Weighted F1:    {weighted_f1:.4f} ({weighted_f1 * 100:.2f}%)")
print()



In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(20, 17))

sns.heatmap(
    cm,
    annot=False,
    cmap="Blues",
    xticklabels=train_dataset.classes,
    yticklabels=train_dataset.classes
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("convnext_tiny — Confusion Matrix")

plt.xticks(
    rotation=90,
    fontsize=8
)

plt.yticks(
    rotation=0,
    fontsize=8
)

plt.tight_layout()

plt.show()

In [ ]:
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=train_dataset.classes,
    digits=4,
    output_dict=True
)

print("=" * 60)
print("PER-CLASS RESULTS")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=train_dataset.classes,
        digits=4
    )
)

In [ ]:
class_f1 = {}

for cls in train_dataset.classes:

    class_f1[cls] = report_dict[cls]["f1-score"]


sorted_classes = sorted(
    class_f1.items(),
    key=lambda x: x[1]
)


print("=" * 60)
print("5 WORST CLASSES")
print("=" * 60)

for cls, score in sorted_classes[:5]:

    print(
        f"{cls}: "
        f"{score:.4f}"
    )


print()

print("=" * 60)
print("5 BEST CLASSES")
print("=" * 60)

for cls, score in sorted_classes[-5:][::-1]:

    print(
        f"{cls}: "
        f"{score:.4f}"
    )

In [ ]:
def predict(image_path):

    model.eval()

    # Load image
    image = Image.open(
        image_path
    ).convert("RGB")

    # Apply validation preprocessing
    image_tensor = test_transform(image)

    # Add batch dimension
    image_tensor = image_tensor.unsqueeze(0)

    # Move to GPU
    image_tensor = image_tensor.to(
        device,
        non_blocking=True
    )

    with torch.no_grad():

        outputs = model(
            image_tensor
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        confidence, predicted = torch.max(
            probabilities,
            dim=1
        )

    predicted_class = train_dataset.classes[
        predicted.item()
    ]

    return (
        predicted_class,
        confidence.item()
    )



In [ ]:

print()
print("=" * 60)
print("convnext_tiny BASELINE COMPLETE")
print("=" * 60)

print(f"Number of classes: {NUM_CLASSES}")
print(f"Training images:   {len(train_dataset)}")
print(f"Test images:       {len(test_dataset)}")
print(f"Accuracy:          {accuracy * 100:.2f}%")
print(f"Macro-F1:          {macro_f1 * 100:.2f}%")
print(f"Weighted F1:       {weighted_f1 * 100:.2f}%")
print(f"Best Val Macro-F1: {best_f1 * 100:.2f}%")

print()
print("Saved model:")
print(save_path)